In [38]:
class Vocab:
    def __init__(self, name):
        self.name = name
        self.roots = []
        self.forms = set()


In [39]:
class Root:
    def __init__(self, root, pos, nounClass=None, plural=None, transitive=False):
        self.root = root
        self.pos = pos
        self.nounClass = nounClass
        self.plural = plural
        self.transitive = transitive
        self.forms = get_forms(self)


In [40]:
def get_forms(fromRoot: Root):
    forms = set()
    root = fromRoot.root
    def getVerbForms(rt: str, can_have_objects):
        verb_forms = set()

        #INFINITIVE 
        verb_forms.add("ku" + rt)
        #HABITUAL
        verb_forms.add("hu" + rt)

        # there are only 11 monosyllabic acting verbs. 
        isMonosyllabic = rt in ["la","fa","ja","nywa","la",'pa','wa','cha','chwa','isha','nya']
        # check for words beginning with vowels 
        startsWithVowel = rt[0] in "aeiou"

        #INDICATIVES and RELATIVES

        #add for looping through; positive verb_forms
        positiveSubjects = ["ni","u","a","tu","wa","m","u","i","li","ya","ki","vi","zi","ku"]
        #covers present, past, future, perfect, perfect already, conditional, and regret
        positiveTenses = ["na","li","ta","me", "mesha","nge","ngali", "ka"]
        #relative markers, including blank which would be the indicative
        relatives = ['','ye','o','cho','vyo','yo','lo','zo','ko','po','mo']

        #affirmative indicatives, and relatives as the affirmative can have relatives. 
        affirmativeRoot = "ku" + rt if isMonosyllabic else rt
        for i in positiveSubjects:
            for j in positiveTenses:
                for k in relatives:
                    verb_forms.add( i + j + k + affirmativeRoot )

        #temporal relative
        for i in positiveSubjects:
            for j in ['li','taka','na']:
                verb_forms.add(i + j + "po" + rt)
    
        
        #add for looping through; negative verb_forms
        negativeSubjects = ['si','hu','ha','hatu','hawa','ham','hau','hai','hali','haya','haki','havi','hazi','haku']
        #present tense
        negativeRoot = rt
        if rt[-1] == "a":
            negativeRoot = rt[:-1] + "i"
        for i in negativeSubjects:
            verb_forms.add(i + negativeRoot)
        #past, future, perfect, conditional, and regret tenses
        negativeTenses = ["ku","ja","ta","singe","singali"]
        for i in negativeSubjects:
            for j in negativeTenses:
                verb_forms.add(i + j + rt)
        
        #SUBJUNCTIVES
        subjunctiveRoot = rt[:-1] + "e" if rt[-1] == "a" else rt
        for i in positiveSubjects:
            verb_forms.add(i + subjunctiveRoot)
        
        #IMPERATIVE 
        #singular, informal
        verb_forms.add(affirmativeRoot)
        #plural formal
        verb_forms.add(rt[:-1] + "eni" if rt[-1] == "a" else rt + "ni")

        #OBJECTS, starting with personal pronouns and moving to noun class infixes.
        if can_have_objects:
            objectInfixes = ["ni","ku","m","wa","tu","ki","vi","u","i","li","ya","zi","ku","pa"]
            if startsWithVowel:
                objectInfixes[2] = "mw"
            for i in objectInfixes:
                verb_forms.update (getVerbForms(i + rt, can_have_objects=False))
        #RECIPROCAL
        
            verb_forms.update(getVerbForms(rt + "na" if rt[-1] == "a" else rt + "ana", can_have_objects=False))

        return verb_forms 
    
    transitive = fromRoot.transitive
    if fromRoot.pos == "verb":
        forms = getVerbForms(root, can_have_objects=transitive)
    
    plural = fromRoot.plural
    if fromRoot.pos == "noun":
        forms = {root, plural, root + "ni", plural + "ni"}

    return forms

In [41]:
kimbia = Root("kimbia","verb",None, None)

In [42]:
kunywa = Root("nywa","verb", nounClass=None, plural=None, transitive=True)
kunywa.forms

{'wangemovinywa',
 'tutachounywa',
 'kingekozinywa',
 'litayozinywa',
 'hazitawanywa',
 'aliyezinywa',
 'tungeyonywana',
 'wakayeunywa',
 'umeyepanywa',
 'kitazounywa',
 'kutapanywa',
 'wameshavyotunywa',
 'mmezokinywa',
 'kingalikomnywa',
 'zinatunywa',
 'kumevyokunywa',
 'kunakunywa',
 'wanapotunywa',
 'litayoyanywa',
 'kiliyotunywa',
 'umezozinywa',
 'yameshayoyanywa',
 'watavyounywa',
 'yatachonywana',
 'waliokunywa',
 'vililinywa',
 'mtaloninywa',
 'tuliyewanywa',
 'limekunywa',
 'linalozinywa',
 'zingalipanywa',
 'hautanywana',
 'mtakomnywa',
 'zingekovinywa',
 'tutamoyanywa',
 'anamonywana',
 'imeshayeunywa',
 'wangalikopanywa',
 'tulivyokinywa',
 'tungalimnywa',
 'nikayokinywa',
 'wangalichovinywa',
 'kimeshakunywa',
 'tutaponinywa',
 'hasingekunywa',
 'kungalikokunywa',
 'sisingalimnywa',
 'yaliotunywa',
 'ulichoinywa',
 'walipokinywa',
 'ameshaoinywa',
 'nimepolinywa',
 'kungalilinywa',
 'limeozinywa',
 'ungeyekunywa',
 'vingalizokinywa',
 'ningelokinywa',
 'zimezinywa',
 'tu

In [43]:
len(kunywa.forms)

17737

In [44]:
def print_verb_form_count(text_root: str, transitive: bool = True):
    root = Root(text_root, "verb", nounClass=None, plural=None, transitive=transitive)
    print(f"{root.root} has {len(root.forms)} forms")
print_verb_form_count("ona", False)
print_verb_form_count("pa", True)

ona has 1258 forms
pa has 17737 forms


In [45]:
negativeSubjects = ["si","hu","ha"]
positiveSubjects = ["ni","u","a","tu","wa","m","u","i","li","ya","ki","vi","zi","ku"]
negativeSubjects += ["ha" + val for idx, val in enumerate(positiveSubjects) if idx > 2]
negativeSubjects

['si',
 'hu',
 'ha',
 'hatu',
 'hawa',
 'ham',
 'hau',
 'hai',
 'hali',
 'haya',
 'haki',
 'havi',
 'hazi',
 'haku']